In [1]:
%load_ext autoreload
%autoreload 2
import sys

sys.path.insert(0, '..')  # nbs/notebook_cache.py
from notebook_cache import NotebookCache

cache = NotebookCache.here()

# Verne, small, step 01: the camera canvas, on T1 and T1-wide

What's new and how shapes flow: `01_canvas_explained.ipynb`. Step 0: `00_end_to_end.ipynb`.

One box opened: the input. Verne (`build_canvas` in `../verne-v0.5/src/verne/transforms.py`) puts its cameras side by side as one wide image, so the video model sees one picture per moment. Here: [top | wrist], 128 × 256 → 8 × 16 = 128 patches through the same patch embedding, positions learned over the whole canvas. Everything else as step 0.

```-
[top | wrist] ─► 16px patches ─► VIDEO transformer (4 layers, 256 wide) ── k,v per layer ─┐
joints ────────► one more token ─┘   runs once per chunk                                   │
noise [16 x 8] ─► ACTION transformer (4 layers, 128 wide) ◄────────────────────────────────┘
                  10 flow steps ─► 16 joint actions
```

Every score: training seeds `TRAIN_SEEDS` × 50 rollout seeds (1000-1049), at n_exec 8 and 16. Step 0 (`00_end_to_end.ipynb`, efb991d): T1 54/150 and 59/150, T1-wide 5/150 and 5/150, close points blind (slopes ≈ 0, miss ≈ 10 cm).

### step 01.1 — the data


In [2]:
import numpy as np
from frozendict import frozendict

from dream_robot.policies.verne.data import canvas, load_cameras, load_table, to_gpu

NAME = 'pick_place_cube_n100'

In [3]:
@cache
def table():
    state, action, episode = load_table(NAME)  # [N, 8], [N, 8], [N]
    return frozendict(state=state, action=action, episode=episode)

table: cached, 1 call(s) stored


In [4]:
@cache
def cameras():
    top, wrist = load_cameras(NAME)  # [N, 3, 128, 128] uint8 each, a few minutes the first time
    return frozendict(top=top, wrist=wrist)

cameras: cached, 1 call(s) stored


In [5]:
data, cams = table(), cameras()
val = np.random.default_rng(0).permutation(100)[:10]  # as dreamzero / uwm2
rest = np.setdiff1d(np.arange(100), val)
split = frozendict(val=val, labelled=np.random.default_rng(1).choice(rest, 10, replace=False))
demos = to_gpu(
    canvas(cams['top'], cams['wrist']),
    data['state'],
    data['action'],
    data['episode'],
    split['labelled'],
)
print({k: len(v) for k, v in split.items()}, tuple(demos.canvas.shape))

{'val': 10, 'labelled': 10} (21295, 3, 128, 256)


### step 01.2 — the model


In [6]:
import torch

from dream_robot.policies.verne.model import Verne

net = Verne()
print(f'{sum(p.numel() for p in net.parameters()) / 1e6:.2f}M params')

3.68M params


### step 01.3 — train (flow matching on the 16-action chunk, 20k steps, ~3 min each), each training seed


In [7]:
import dream_robot.policies.verne as verne


@cache
def train_run(steps: int, seed: int):
    return verne.train.train(demos, split['labelled'], split['val'], steps, seed=seed)

train_run: cached, 3 call(s) stored


In [8]:
TRAIN_SEEDS = (0, 1, 2)  # results are cached per seed, so added seeds just join

In [9]:
runs = {seed: train_run(20000, seed) for seed in TRAIN_SEEDS}  # seed -> {'weights', 'log'}

step      0   train 2.224   val 2.868
step   1000   train 0.164   val 0.193
step   2000   train 0.079   val 0.165
step   3000   train 0.078   val 0.219
step   4000   train 0.043   val 0.183
step   5000   train 0.046   val 0.201
step   6000   train 0.068   val 0.231
step   7000   train 0.037   val 0.219
step   8000   train 0.039   val 0.203
step   9000   train 0.025   val 0.219
step  10000   train 0.039   val 0.245
step  11000   train 0.034   val 0.256
step  12000   train 0.033   val 0.224
step  13000   train 0.033   val 0.273
step  14000   train 0.043   val 0.265
step  15000   train 0.034   val 0.251
step  16000   train 0.027   val 0.272
step  17000   train 0.021   val 0.251
step  18000   train 0.017   val 0.267
step  19000   train 0.036   val 0.267
step  20000   train 0.012   val 0.304
step      0   train 2.371   val 3.240
step   1000   train 0.088   val 0.199
step   2000   train 0.049   val 0.186
step   3000   train 0.086   val 0.206
step   4000   train 0.058   val 0.189
step   5000 

### step 01.4 — rollouts: T1 seeds 1000-1049, replan after 8 or all 16 actions, each training seed


In [10]:
from dream_robot.policies.verne import rollouts


def total(scores: dict, name: str) -> None:
    # scores: (train seed, n_exec) -> rollout results; prints one line per n_exec
    for n_exec in (8, 16):
        oks = [sum(ok for ok, _, _ in scores[seed, n_exec].values()) for seed in TRAIN_SEEDS]
        n = len(rollouts.SEEDS) * len(TRAIN_SEEDS)
        print(f'{name} n_exec {n_exec:2d}: {" + ".join(map(str, oks))} = {sum(oks)}/{n}')

In [11]:
@cache(2)  # 2: one torch seed per episode, workers
def sweep(seed: int, n_exec: int) -> dict[int, tuple[bool, str, int]]:
    return rollouts.sweep(runs[seed]['weights'], demos, wide=False, n_exec=n_exec)

sweep: cached, 6 call(s) stored


In [12]:
scores = {(seed, n): sweep(seed, n) for seed in TRAIN_SEEDS for n in (8, 16)}
total(scores, 'T1')

n_exec  8: 10/50   failures {'no_grasp': 40}
n_exec 16: 15/50   failures {'no_grasp': 35}
n_exec  8: 30/50   failures {'no_grasp': 18, 'wrong_target': 2}
n_exec 16: 16/50   failures {'no_grasp': 30, 'dropped': 2, 'wrong_target': 2}
n_exec  8: 19/50   failures {'no_grasp': 28, 'wrong_target': 3}
n_exec 16: 17/50   failures {'no_grasp': 33}
T1 n_exec  8: 10 + 30 + 19 = 59/150
T1 n_exec 16: 15 + 16 + 17 = 48/150


**T1, train seeds 0 + 1 + 2: 10 + 30 + 19 = 59/150 at n_exec 8, 15 + 16 + 17 = 48/150 at 16** (step 0: 56 and 54). No gain on T1, where the cube barely moves. Seeds spread 10-30 of 50; seed 1 at n_exec 8 (30/50) is the best single run so far. Held-out loss ends worse than step 0 (0.304, 0.255, 0.239 vs 0.246, 0.196, 0.201): twice the tokens on the same 10 demos.

### step 01.5 — T1-wide: a cube that moves (x ±10 cm, y −10..+6 cm)

On T1 the cube spawns within ±3 cm, so a policy can grasp the average spot without looking. T1-wide (`data/robosuite/pick_place_cube_wide_n100`, expert 100/100) makes it find the cube. 90 train demos, 10 held out (the same episode ids as T1's val), 20k steps, each training seed, rollout seeds 1000-1049.


In [13]:
WIDE = 'pick_place_cube_wide_n100'


@cache
def table_wide():
    state, action, episode = load_table(WIDE)  # [N, 8], [N, 8], [N]
    return frozendict(state=state, action=action, episode=episode)

table_wide: cached, 1 call(s) stored


In [14]:
@cache
def cameras_wide():
    top, wrist = load_cameras(WIDE)  # [N, 3, 128, 128] uint8 each, a few minutes the first time
    return frozendict(top=top, wrist=wrist)

cameras_wide: cached, 1 call(s) stored


In [15]:
w_data, w_cams = table_wide(), cameras_wide()
w_split = frozendict(val=val, train=np.setdiff1d(np.arange(100), val))
w_demos = to_gpu(
    canvas(w_cams['top'], w_cams['wrist']),
    w_data['state'],
    w_data['action'],
    w_data['episode'],
    w_split['train'],
)
print({k: len(v) for k, v in w_split.items()}, tuple(w_demos.canvas.shape))

{'val': 10, 'train': 90} (22409, 3, 128, 256)


In [16]:
@cache
def train_wide(steps: int, seed: int):
    return verne.train.train(w_demos, w_split['train'], w_split['val'], steps, seed=seed)

train_wide: cached, 3 call(s) stored


In [17]:
w_runs = {seed: train_wide(20000, seed) for seed in TRAIN_SEEDS}

step      0   train 2.231   val 2.667
step   1000   train 0.190   val 0.134
step   2000   train 0.078   val 0.100
step   3000   train 0.102   val 0.103
step   4000   train 0.047   val 0.078
step   5000   train 0.050   val 0.068
step   6000   train 0.073   val 0.079
step   7000   train 0.042   val 0.062
step   8000   train 0.050   val 0.071
step   9000   train 0.029   val 0.064
step  10000   train 0.053   val 0.069
step  11000   train 0.042   val 0.066
step  12000   train 0.044   val 0.055
step  13000   train 0.047   val 0.067
step  14000   train 0.060   val 0.053
step  15000   train 0.042   val 0.050
step  16000   train 0.034   val 0.049
step  17000   train 0.030   val 0.058
step  18000   train 0.032   val 0.047
step  19000   train 0.038   val 0.054
step  20000   train 0.025   val 0.063
step      0   train 2.318   val 3.291
step   1000   train 0.120   val 0.135
step   2000   train 0.082   val 0.101
step   3000   train 0.102   val 0.096
step   4000   train 0.078   val 0.085
step   5000 

In [18]:
@cache(2)  # 2: one torch seed per episode, workers
def sweep_wide(seed: int, n_exec: int) -> dict[int, tuple[bool, str, int]]:
    return rollouts.sweep(w_runs[seed]['weights'], w_demos, wide=True, n_exec=n_exec)

sweep_wide: cached, 5 call(s) stored


In [19]:
w_scores = {(seed, n): sweep_wide(seed, n) for seed in TRAIN_SEEDS for n in (8, 16)}
total(w_scores, 'T1-wide')

n_exec  8:  3/50   failures {'no_grasp': 46, 'dropped': 1}
n_exec 16:  3/50   failures {'no_grasp': 47}
n_exec  8:  3/50   failures {'no_grasp': 42, 'dropped': 4, 'wrong_target': 1}
n_exec 16: 10/50   failures {'no_grasp': 40}
n_exec  8: 12/50   failures {'no_grasp': 37, 'dropped': 1}
n_exec 16:  8/50   failures {'no_grasp': 42}
T1-wide n_exec  8: 3 + 3 + 12 = 18/150
T1-wide n_exec 16: 3 + 10 + 8 = 21/150


**T1-wide, train seeds 0 + 1 + 2: 3 + 3 + 12 = 18/150 at n_exec 8, 3 + 10 + 8 = 21/150 at 16** (step 0: 4 and 3). The canvas helps where the cube moves: 39 vs 7 over both n_exec. Seed 2 carries most of it. Held-out loss 0.063, 0.049, 0.047 (step 0: 0.055, 0.050, 0.047): the loss doesn't see the difference.

### step 01.6 — the close-point probe: does the gripper close where the cube is?

For each T1-wide rollout (n_exec 8): the gripper's xy when it first commands close, against the cube's spawn xy. Slope ≈ 1 and a small miss: it goes to the cube. Slope ≈ 0: it goes to a remembered spot whatever the cube does. One line per training seed.


In [20]:
@cache(2)  # 2: one torch seed per episode, workers
def close_points_wide(seed: int, n_exec: int) -> dict:
    return rollouts.close_points(w_runs[seed]['weights'], w_demos, wide=True, n_exec=n_exec)

close_points_wide: cached, 2 call(s) stored


In [21]:
fits = {}
for seed in TRAIN_SEEDS:
    print(f'train seed {seed}: ', end='')
    fits[seed] = rollouts.close_fit(close_points_wide(seed, 8))

train seed 0: closes 50/50, miss  9.4 cm, x slope +0.25 (corr +0.24, intercept +3.0 cm), y slope +0.07 (corr +0.09, intercept -8.0 cm)
train seed 1: closes 50/50, miss 10.2 cm, x slope +0.03 (corr +0.03, intercept +7.0 cm), y slope +0.02 (corr +0.03, intercept -3.2 cm)
train seed 2: closes 50/50, miss  7.5 cm, x slope +0.61 (corr +0.45, intercept +4.6 cm), y slope +0.30 (corr +0.31, intercept -3.6 cm)


**Probe: only some runs find the cube.**
- seed 0: x slope +0.25, y +0.07; intercepts x +3.0, y −8.0 cm; mean miss 9.4 cm
- seed 1: x slope +0.03, y +0.02; intercepts x +7.0, y −3.2 cm; mean miss 10.2 cm
- seed 2: x slope **+0.61** (corr 0.45), y +0.30; intercepts x +4.6, y −3.6 cm; mean miss **7.5 cm**

Step 0's three seeds were all blind (slopes ≈ 0, miss ≈ 10 cm). Seed 0 trained on the local 4060 instead of this pod's 4090, same code and bit-identical to the first pass, gives x slope **+0.95**, miss 3.8 cm (`experiments/verne/notes.md`): the same seed lands on a seeing or a blind model depending on the GPU. Finding the cube is fragile; 2 of the 4 canvas runs found it.